# Copilot Medallion — orchestrator

Deployed by `copilot.roesli.org`. Reads parameters injected by the Fabric Job Scheduler API and builds Bronze/Silver/Gold inside the target lakehouse using ABFS paths (no lakehouse attachment required).

In [ ]:
# default parameters (overridden by Job Scheduler)
source_lakehouse_id = ""
source_tables_csv = ""
target_lakehouse_id = ""
target_lakehouse_name = ""
workspace_id = ""
run_id = "local-dev"
spec_url = ""

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException
from datetime import datetime

tables = [t.strip() for t in source_tables_csv.split(',') if t.strip()]
src_base = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{source_lakehouse_id}"
tgt_base = f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{target_lakehouse_id}"

print(f"run_id={run_id}")
print(f"source={src_base}")
print(f"target={tgt_base}")
print(f"tables={tables}")
print(f"spec={spec_url}")

## Bronze — raw copies

In [ ]:
bronze_results = {}
for t in tables:
    src_path = f"{src_base}/Tables/{t}"
    tgt_path = f"{tgt_base}/Tables/bronze_{t.lower()}"
    try:
        df = spark.read.format('delta').load(src_path)
        df.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(tgt_path)
        bronze_results[t] = df.count()
        print(f"  bronze_{t.lower()} rows={bronze_results[t]}")
    except AnalysisException as e:
        bronze_results[t] = f"ERROR: {e}"
        print(f"  {t}: {e}")

## Silver — cleaned typed

In [ ]:
silver_results = {}
ts = F.lit(datetime.utcnow().isoformat())
for t in tables:
    if isinstance(bronze_results.get(t), str):  # error
        continue
    bronze_path = f"{tgt_base}/Tables/bronze_{t.lower()}"
    silver_path = f"{tgt_base}/Tables/silver_{t.lower()}"
    df = spark.read.format('delta').load(bronze_path)
    # trim strings, drop fully-null columns
    null_counts = df.select([F.sum(F.col(c).isNull().cast('int')).alias(c) for c in df.columns]).collect()[0].asDict()
    keep = [c for c, n in null_counts.items() if n < df.count()]
    df2 = df.select(*[F.trim(F.col(c)).alias(c) if str(df.schema[c].dataType).startswith('StringType') else F.col(c) for c in keep])
    df2 = df2.withColumn('_silver_ts', ts)
    df2.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(silver_path)
    silver_results[t] = df2.count()
    print(f"  silver_{t.lower()} rows={silver_results[t]}")

## Gold — star schema if AdventureWorksLT-shaped

In [ ]:
lc = {t.lower(): t for t in tables}
awl_keys = {'customer','product','salesorderheader','salesorderdetail'}
gold_built = False
if awl_keys.issubset(set(lc.keys())):
    cust = spark.read.format('delta').load(f"{tgt_base}/Tables/silver_customer")
    prod = spark.read.format('delta').load(f"{tgt_base}/Tables/silver_product")
    soh  = spark.read.format('delta').load(f"{tgt_base}/Tables/silver_salesorderheader")
    sod  = spark.read.format('delta').load(f"{tgt_base}/Tables/silver_salesorderdetail")
    cust.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(f"{tgt_base}/Tables/dim_customer")
    prod.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(f"{tgt_base}/Tables/dim_product")
    # date dim
    dates = soh.select(F.to_date('OrderDate').alias('Date')).distinct().filter('Date is not null')
    dates = (dates
             .withColumn('Year', F.year('Date'))
             .withColumn('Quarter', F.quarter('Date'))
             .withColumn('Month', F.month('Date'))
             .withColumn('Day', F.dayofmonth('Date')))
    dates.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(f"{tgt_base}/Tables/dim_date")
    # fact
    fact = (sod.alias('d')
            .join(soh.alias('h'), F.col('d.SalesOrderID') == F.col('h.SalesOrderID'), 'inner')
            .select(
                F.col('h.CustomerID').alias('CustomerKey'),
                F.col('d.ProductID').alias('ProductKey'),
                F.to_date('h.OrderDate').alias('DateKey'),
                F.col('d.OrderQty').alias('Qty'),
                F.col('d.UnitPrice').alias('UnitPrice'),
                F.col('d.LineTotal').alias('LineTotal')))
    fact.write.format('delta').mode('overwrite').option('overwriteSchema','true').save(f"{tgt_base}/Tables/fact_sales")
    gold_built = True
print(f"gold_built={gold_built}")

In [ ]:
print({'run_id': run_id, 'bronze': bronze_results, 'silver': silver_results, 'gold_built': gold_built})